# Session 11 — Your turn: Attention

Attention is three matrix products and a softmax. You will write it for a three-word example, then add the causal mask a text generator needs.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np

tokens = ["robot", "battery", "it"]
Q = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 2.0]])      # what each token asks for
K = np.array([[1.0, 0.0], [0.0, 1.0], [0.0, 0.2]])      # what each token offers to be found by
V = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])      # what each token passes on when it is chosen
from aij_check import check, close, summary

## Task 1: Attention by hand

Write `softmax(scores)` (over the last axis, subtracting the row maximum first for stability) and compute `scores = Q @ K.T / sqrt(d)` with `d` the number of columns of `K`,
`weights = softmax(scores)`, and `output = weights @ V`. Then set `it_looks_at` to the token that "it" (the last row) gives the largest weight.

In [2]:
def softmax(s):
    e = np.exp(s - s.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


d = K.shape[1]
scores = Q @ K.T / np.sqrt(d)
weights = softmax(scores)
output = weights @ V
it_looks_at = tokens[int(weights[2].argmax())]
print(np.round(weights, 2), it_looks_at)

[[0.5  0.25 0.25]
 [0.24 0.49 0.28]
 [0.21 0.6  0.19]] battery


In [3]:
check("every row of weights sums to 1", lambda: close(weights.sum(axis=1), 1.0, 1e-9))
check("the weights match the reference", lambda: close(weights, __import__("scipy.special", fromlist=["softmax"]).softmax(Q @ K.T / np.sqrt(2), axis=1), 1e-9),
      "scores = Q @ K.T / sqrt(d), then softmax along each row.")
check("'it' looks mostly at 'battery'", lambda: it_looks_at == "battery")
check("the output has one new vector per token", lambda: np.shape(output) == (3, 2) and close(output, weights @ V, 1e-9))
summary()

PASS  every row of weights sums to 1


PASS  the weights match the reference
PASS  'it' looks mostly at 'battery'
PASS  the output has one new vector per token

4 of 4 checks passed.  Well done.


**Why it works.** Each row of `weights` says how much one token reads from every token; `output` mixes the value vectors in those proportions. Dividing by sqrt(d) keeps the scores from growing with the vector size, so the softmax does not collapse onto one token.

## Task 2: The causal mask

A model that writes text must not read the future. Build `mask` (True where a token may look: itself and earlier tokens), set the scores of forbidden positions to `-1e9` before the softmax, and compute `causal_weights`.
Then store in `first_token_row` the weights of the first token, and in `leaks` whether any weight above the diagonal is larger than 1e-6.

In [4]:
mask = np.tril(np.ones((3, 3), dtype=bool))
causal_scores = np.where(mask, scores, -1e9)
causal_weights = softmax(causal_scores)
first_token_row = causal_weights[0]
leaks = bool((causal_weights[np.triu_indices(3, 1)] > 1e-6).any())
print(np.round(causal_weights, 2))

[[1.   0.   0.  ]
 [0.33 0.67 0.  ]
 [0.21 0.6  0.19]]


In [5]:
check("the mask is lower-triangular", lambda: np.array_equal(mask, np.tril(np.ones((3, 3), dtype=bool))))
check("rows still sum to 1", lambda: close(causal_weights.sum(axis=1), 1.0, 1e-9))
check("the first token can only read itself", lambda: close(first_token_row, [1, 0, 0], 1e-9))
check("no token reads a later one", lambda: leaks is False)
summary()

PASS  the mask is lower-triangular
PASS  rows still sum to 1
PASS  the first token can only read itself
PASS  no token reads a later one

4 of 4 checks passed.  Well done.


**Why it works.** Setting a score to a huge negative number makes its softmax weight 0 and renormalises the rest, so each row still sums to 1. The first token has only itself to read, which is why generation has to start from a prompt.